# 主变压器状态评价模型的跨电压等级样本评价。 装有 Python3.x（含 pandas、scikit-learn 库）的计算机 1 台、开发环境（JupyterNotebook 或 · 教学参考

这是帮助理解写法的参考实现，不是官方标准答案。请在 `answer.ipynb` 独立作答；不要在本文件中覆盖自己的草稿。

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

data = pd.read_csv("04/SS-5-5-5-02_transformer_health.csv")
data = data.dropna().drop_duplicates().reset_index(drop=True)
X = data[["运行年限", "油温", "绕组温度", "负载率", "H2含量", "总烃含量", "油中含水量", "介质损耗", "历史故障次数", "电压等级"]]
y = data["是否需检修"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2026, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
model = RandomForestClassifier(n_estimators=100, random_state=2026)
model.fit(X_train_s, y_train)
pred = model.predict(X_test_s)
test_info = X_test.copy()
test_info["真实标签"] = y_test.values
test_info["预测标签"] = pred
counts = {}
accs = {}
for level in [110, 220, 500]:
    sub = test_info[test_info["电压等级"] == level]
    counts[level] = len(sub)
    accs[level] = accuracy_score(sub["真实标签"], sub["预测标签"]) if len(sub) else 0
    print(level, counts[level], accs[level])
print("重点关注等级：", min(counts, key=counts.get))
